# Maia Lite 0.3.0 — A100 sustained throughput & compute-budget benchmark

**Goal:** Compare the 304.5M architecture against a ~0.5B candidate, with microbatches 1/2/4/8/16, 2048-token context, optional activation checkpointing, real optimizer updates, sustained throughput, and projected time/cost against **3× the measured Maia Lite 0.2.0 baseline**.

**Important:** A smoke-test step is not a production throughput estimate. This notebook benchmarks actual forward + backward + optimizer updates (not just inference), reports warmup and measurement separately, and uses synchronized CUDA timings. A 0.5B model or a second corpus pass is *not* automatically more compute-efficient. Chinchilla ~20 training tokens per parameter is a heuristic, not a guarantee of optimality.

**Safety:** Synthetic token IDs only; never modifies the corpus. Outputs to a unique timestamped directory. Tests can be slow and consume GPU credits. Begin with the default limited sweep. Stop when sufficient evidence is collected.

In [ ]:
import os, sys, math, json, time, gc, platform, hashlib
from pathlib import Path
from datetime import datetime, timezone
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda)
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is required. In Colab: Runtime > Change runtime type > A100 GPU.')
DEVICE = torch.device('cuda')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
print('BF16 supported:', torch.cuda.is_bf16_supported())
assert torch.cuda.is_bf16_supported(), 'This notebook requires BF16-capable CUDA GPU.'
torch.manual_seed(20261027)
torch.cuda.manual_seed_all(20261027)

## Experiment configuration

Set `BASELINE_DAYS` to the **measured wall-clock days** for the earlier Maia Lite 0.2.0 run, and `BASELINE_TOKENS` to the **actual tokens consumed**, not dataset size. The default `None` means that the 3× affordability decision is **not** evaluated. If that earlier run used multiple GPUs, set the matching `BASELINE_GPU_COUNT` and compare GPU-days, not wall-clock alone.

For fast initial screening, keep `MODEL_KEYS=['small']`, `CHECKPOINT_MODES=[False]`. Then enable the ~0.5B candidate and activation checkpointing. Each combination performs several real optimizer updates. The `max_seconds_per_case` guard is approximate (it checks between steps).

In [ ]:
# ---- Editable experiment inputs ----
BASELINE_DAYS = None           # Example: 2.8 (replace with actual measured time)
BASELINE_TOKENS = None         # Example: 20_000_000_000 (replace with actual consumed tokens)
BASELINE_GPU_COUNT = 1        # Number of A100-equivalent GPUs in the baseline run
CURRENT_GPU_COUNT = 1
PRICE_USD_PER_GPU_HOUR = None # Optional actual provider rate; do not assume a rate

CORPUS_RAW_TOKENS = 7_590_554_035
CORPUS_DOCUMENTS = 10_101_231
EOS_PER_DOCUMENT = 1
SEQ_LEN = 2048
VOCAB_SIZE = 50257
BATCH_SIZES = [1, 2, 4, 8, 16]
MODEL_KEYS = ['small', 'half_b']  # Set ['small'] for a shorter first pass
CHECKPOINT_MODES = [False, True] # Set [False] initially
WARMUP_UPDATES = 2
MEASURE_UPDATES = 4
MAX_SECONDS_PER_CASE = 180
TARGET_EFFECTIVE_TOKENS = 32768
ENABLE_CUDNN_BENCHMARK = False

# Run isolation: never overwrite a previous benchmark.
OUTPUT_ROOT = Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/corpus_v8/runs/acquisition_v8_5_001/architecture_benchmark_v2')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
OUTPUT = OUTPUT_ROOT / RUN_ID
OUTPUT.mkdir(parents=True, exist_ok=False)
print('Output:', OUTPUT)

## Decoder-only reference implementation

The **small** configuration matches the architectural hyperparameters from the previous smoke test (20 layers, width 1024, 16 heads, FFN 2752, tied embeddings, RoPE/RMSNorm/SwiGLU). The **half_b** configuration is a *candidate*, not a previously approved architecture; its exact parameter count is calculated below. No claim is made that the implementation is kernel-identical to the prior notebook. This is a comparative compute benchmark, not a bitwise reproducibility test against that implementation.

The benchmark uses PyTorch SDPA causal attention and full vocabulary cross-entropy. To avoid holding large logits in GPU memory, the loss is computed in time chunks; this changes memory usage and may affect throughput, so production training must reproduce the same loss strategy or be benchmarked separately.

In [ ]:
from dataclasses import dataclass, asdict
@dataclass(frozen=True)
class Config:
    layers: int
    width: int
    heads: int
    ffn: int
    kv_heads: int
    vocab: int = VOCAB_SIZE
    context: int = SEQ_LEN
    rope_theta: float = 10000.0
    eps: float = 1e-5

CONFIGS = {
    'small': Config(layers=20, width=1024, heads=16, kv_heads=16, ffn=2752),
    'half_b': Config(layers=28, width=1152, heads=18, kv_heads=18, ffn=3072),
}

class RMSNorm(nn.Module):
    def __init__(self, dim, eps):
        super().__init__(); self.weight=nn.Parameter(torch.ones(dim)); self.eps=eps
    def forward(self, x):
        return F.rms_norm(x, (x.shape[-1],), self.weight, self.eps)

class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c=c; self.norm1=RMSNorm(c.width,c.eps); self.norm2=RMSNorm(c.width,c.eps)
        self.q=nn.Linear(c.width,c.width,bias=False)
        self.k=nn.Linear(c.width,c.width,bias=False)
        self.v=nn.Linear(c.width,c.width,bias=False)
        self.o=nn.Linear(c.width,c.width,bias=False)
        self.gate=nn.Linear(c.width,c.ffn,bias=False)
        self.up=nn.Linear(c.width,c.ffn,bias=False)
        self.down=nn.Linear(c.ffn,c.width,bias=False)
    def forward(self,x,cos,sin):
        b,t,d=x.shape; h=self.c.heads; hd=d//h
        y=self.norm1(x)
        q=self.q(y).view(b,t,h,hd).transpose(1,2)
        k=self.k(y).view(b,t,h,hd).transpose(1,2)
        v=self.v(y).view(b,t,h,hd).transpose(1,2)
        def rotate(z):
            a=z[...,::2]; b_=z[...,1::2]
            return torch.stack((-b_,a),dim=-1).flatten(-2)
        q=q*cos+rotate(q)*sin
        k=k*cos+rotate(k)*sin
        att=F.scaled_dot_product_attention(q,k,v,is_causal=True,dropout_p=0.0)
        x=x+self.o(att.transpose(1,2).contiguous().view(b,t,d))
        y=self.norm2(x)
        return x+self.down(F.silu(self.gate(y))*self.up(y))

class MaiaDecoder(nn.Module):
    def __init__(self,c,gradient_checkpointing=False):
        super().__init__(); self.c=c; self.gc=gradient_checkpointing
        self.embed=nn.Embedding(c.vocab,c.width)
        self.blocks=nn.ModuleList([Block(c) for _ in range(c.layers)])
        self.norm=RMSNorm(c.width,c.eps)
        self.head=nn.Linear(c.width,c.vocab,bias=False)
        self.head.weight=self.embed.weight
        hd=c.width//c.heads
        inv=1.0/(c.rope_theta**(torch.arange(0,hd,2,dtype=torch.float32)/hd))
        self.register_buffer('inv_freq',inv,persistent=False)
    def forward(self,ids):
        b,t=ids.shape
        pos=torch.arange(t,device=ids.device,dtype=torch.float32)
        ang=torch.outer(pos,self.inv_freq.to(ids.device))
        emb=torch.repeat_interleave(ang,2,dim=-1)
        cos=emb.cos()[None,None,:,:].to(self.embed.weight.dtype)
        sin=emb.sin()[None,None,:,:].to(self.embed.weight.dtype)
        x=self.embed(ids)
        for block in self.blocks:
            if self.gc and self.training:
                x=checkpoint(block,x,cos,sin,use_reentrant=False)
            else:
                x=block(x,cos,sin)
        return self.norm(x)

# Compute cross entropy without materializing full B x T x V logits at once.
def chunked_causal_loss(model, hidden, ids, time_chunk=128):
    b,t,d=hidden.shape
    total=hidden.new_zeros((),dtype=torch.float32)
    count=0
    for start in range(0,t-1,time_chunk):
        end=min(t-1,start+time_chunk)
        logits=F.linear(hidden[:,start:end,:],model.embed.weight).float()
        labels=ids[:,start+1:end+1]
        total=total+F.cross_entropy(logits.reshape(-1,model.c.vocab),labels.reshape(-1),reduction='sum')
        count+=labels.numel()
    return total/count

for name,c in CONFIGS.items():
    m=MaiaDecoder(c)
    p=sum(v.numel() for v in m.parameters())
    print(name, f'{p:,} parameters', 'config:',asdict(c))
    del m

## Sustained benchmark: real optimizer updates

A benchmark update processes `microbatch × context × accumulation` input tokens. We count **predicted** tokens as `microbatch × (context − 1) × accumulation` for throughput. The optimizer is AdamW, with BF16 model weights and float32 gradients where PyTorch naturally retains them; this is a **benchmark setup**, not a fully tuned production optimizer. A different mixed-precision/optimizer strategy may change throughput and memory. Measurements include optimizer updates and gradient zeroing, but **exclude data loading and checkpoint I/O**.

For each case, optimizer steps are timed with CUDA synchronization. CUDA OOM is caught and recorded; other errors are recorded, not silently marked PASS.

In [ ]:
def cleanup():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

def run_case(name,c,batch,gc_enabled):
    cleanup()
    rec={'model':name,'batch':batch,'checkpointing':gc_enabled,'status':'NOT_RUN',
         'config':asdict(c)}
    model=optimizer=ids=None
    try:
        model=MaiaDecoder(c,gradient_checkpointing=gc_enabled).to(DEVICE,dtype=torch.bfloat16)
        rec['parameters']=sum(p.numel() for p in model.parameters())
        model.train()
        optimizer=torch.optim.AdamW(model.parameters(),lr=1e-4,weight_decay=0.01,foreach=True)
        ids=torch.randint(0,c.vocab,(batch,c.context),device=DEVICE,dtype=torch.long)
        accum=max(1,math.ceil(TARGET_EFFECTIVE_TOKENS/(batch*c.context)))
        rec['accumulation']=accum
        rec['input_tokens_per_update']=batch*c.context*accum
        rec['predicted_tokens_per_update']=batch*(c.context-1)*accum
        times=[]; losses=[]; peak_alloc=0; peak_reserved=0
        for step in range(WARMUP_UPDATES+MEASURE_UPDATES):
            torch.cuda.synchronize()
            if step==WARMUP_UPDATES:
                torch.cuda.reset_peak_memory_stats()
            start=time.perf_counter()
            optimizer.zero_grad(set_to_none=True)
            for _ in range(accum):
                hidden=model(ids)
                loss=chunked_causal_loss(model,hidden,ids)
                (loss/accum).backward()
            optimizer.step()
            torch.cuda.synchronize()
            dt=time.perf_counter()-start
            if step>=WARMUP_UPDATES:
                times.append(dt);losses.append(float(loss.detach().item()))
                peak_alloc=max(peak_alloc,torch.cuda.max_memory_allocated())
                peak_reserved=max(peak_reserved,torch.cuda.max_memory_reserved())
                if sum(times)>MAX_SECONDS_PER_CASE:
                    break
        if not times or not all(math.isfinite(x) for x in losses):
            raise RuntimeError('No valid measured optimizer updates or non-finite loss')
        elapsed=sum(times)
        rec.update(status='PASS',measured_updates=len(times),seconds_per_update_mean=elapsed/len(times),
                   seconds_per_update_median=sorted(times)[len(times)//2],
                   predicted_tokens_per_second=rec['predicted_tokens_per_update']*len(times)/elapsed,
                   input_tokens_per_second=rec['input_tokens_per_update']*len(times)/elapsed,
                   loss_last=losses[-1],peak_allocated_gib=peak_alloc/2**30,
                   peak_reserved_gib=peak_reserved/2**30,measured_step_seconds=times)
    except torch.cuda.OutOfMemoryError as e:
        rec.update(status='OOM',error=str(e)[:400])
    except Exception as e:
        rec.update(status='ERROR',error=f'{type(e).__name__}: {str(e)[:400]}')
    finally:
        del ids,optimizer,model
        cleanup()
    return rec

In [ ]:
RESULTS=[]
for name in MODEL_KEYS:
    c=CONFIGS[name]
    for mode in CHECKPOINT_MODES:
        for bs in BATCH_SIZES:
            print(f'Running {name} checkpointing={mode} batch={bs}',flush=True)
            rec=run_case(name,c,bs,mode)
            RESULTS.append(rec)
            print(' ',rec['status'],'tokens/s=',round(rec.get('predicted_tokens_per_second',0)),
                  'VRAM GiB=',round(rec.get('peak_reserved_gib',0),2),
                  rec.get('error','')[:120],flush=True)
            # Persist partial results after every case for interrupted Colab sessions.
            (OUTPUT/'partial_results.json').write_text(json.dumps(RESULTS,indent=2,ensure_ascii=False),encoding='utf-8')
print('Completed cases:',len(RESULTS))

## Compute and budget analysis

The classical Chinchilla rule-of-thumb is **~20 training tokens per parameter**. The corpus contains 7.5906B raw audited tokens; with one EOS per document it becomes ~7.6007B before any exclusions or split. The number of *training* tokens after holdouts is lower. This notebook displays the simple full-corpus arithmetic for planning only, and does **not** claim that twice through the same corpus yields the same information gain as twice as much distinct text.

A model with 0.5B parameters has a ~10B-token heuristic target, about **1.32 corpus passes**. Two passes would be ~15.2B tokens, or ~30.4 tokens/parameter, **above** the 20:1 heuristic. For the ~304M model, one pass already supplies ~25 tokens/parameter. The optimal tradeoff depends on compute, data quality and observed validation losses.

**Budget:** Compare projected GPU-days against `3 × BASELINE_DAYS × BASELINE_GPU_COUNT`. The baseline is not assumed; enter your real historical measurements. Projection from synthetic benchmarks remains provisional until sustained real-data throughput is measured.

In [ ]:
from IPython.display import display
import pandas as pd
rows=[]
for r in RESULTS:
    rows.append({k:r.get(k) for k in ['model','batch','checkpointing','status','parameters',
        'accumulation','predicted_tokens_per_second','seconds_per_update_mean','peak_reserved_gib','error']})
df=pd.DataFrame(rows)
display(df)
valid=[r for r in RESULTS if r['status']=='PASS']
fastest={}
for r in valid:
    if r['model'] not in fastest or r['predicted_tokens_per_second']>fastest[r['model']]['predicted_tokens_per_second']:
        fastest[r['model']]=r

# The raw corpus estimate includes a planning-only EOS allowance; held-out sets reduce train tokens.
planning_tokens_per_pass=CORPUS_RAW_TOKENS+CORPUS_DOCUMENTS*EOS_PER_DOCUMENT
budget_gpu_days=None
if BASELINE_DAYS is not None:
    budget_gpu_days=3*BASELINE_DAYS*BASELINE_GPU_COUNT

summary=[]
for name,c in CONFIGS.items():
    # Calculate exact count independently of whether the model was benchmarked.
    tmp=MaiaDecoder(c)
    params=sum(p.numel() for p in tmp.parameters()); del tmp
    heuristic_tokens=20*params
    best=fastest.get(name)
    for passes in [1,2]:
        tokens=planning_tokens_per_pass*passes
        tps=best['predicted_tokens_per_second'] if best else None
        gpu_days=tokens/tps/86400*CURRENT_GPU_COUNT if tps else None
        # For a single GPU this equals wall days. For multiple GPUs throughput must be measured separately.
        affordable=(gpu_days<=budget_gpu_days) if gpu_days is not None and budget_gpu_days is not None else None
        summary.append({'model':name,'params':params,'passes':passes,
            'tokens_planned':tokens,'tokens_per_param':tokens/params,
            'chinchilla_target_tokens':heuristic_tokens,
            'best_batch':best['batch'] if best else None,
            'best_checkpointing':best['checkpointing'] if best else None,
            'measured_predicted_tps':tps,
            'projected_gpu_days_at_measured_tps':gpu_days,
            'baseline_3x_gpu_day_cap':budget_gpu_days,
            'within_3x_baseline':affordable,
            'estimated_cost_usd':gpu_days*24*PRICE_USD_PER_GPU_HOUR if gpu_days is not None and PRICE_USD_PER_GPU_HOUR is not None else None})
summary_df=pd.DataFrame(summary)
display(summary_df)
print('WARNING: These projections exclude I/O, evaluation, checkpointing, data exclusions, and tuning.')
if BASELINE_DAYS is None:
    print('BUDGET STATUS: UNDETERMINED — set BASELINE_DAYS and rerun this cell.')

## Save reproducible report and decision gates

A PASS means this *synthetic* configuration ran. It does **not** certify a production training time. Recommended gates before selecting a final architecture: (1) ≥20 sustained measured optimizer updates on real packed data; (2) baseline cost verified in comparable GPU-days; (3) held-out validation loss by language; (4) GPU checkpoint resume regression; (5) full production memory headroom. A 3× cost limit should be treated as a hard constraint when choosing model size and training tokens.

In [ ]:
report={
 'schema_version':'2.0',
 'purpose':'A100 sustained synthetic throughput and compute-budget planning',
 'created_utc':datetime.now(timezone.utc).isoformat(),
 'run_id':RUN_ID,
 'environment':{'python':platform.python_version(),'torch':torch.__version__,
     'cuda':torch.version.cuda,'gpu':torch.cuda.get_device_name(0)},
 'settings':{'baseline_days':BASELINE_DAYS,'baseline_tokens':BASELINE_TOKENS,
     'baseline_gpu_count':BASELINE_GPU_COUNT,'current_gpu_count':CURRENT_GPU_COUNT,
     'usd_per_gpu_hour':PRICE_USD_PER_GPU_HOUR,'seq_len':SEQ_LEN,
     'batch_sizes':BATCH_SIZES,'checkpoint_modes':CHECKPOINT_MODES,
     'warmup_updates':WARMUP_UPDATES,'measure_updates':MEASURE_UPDATES,
     'target_effective_tokens':TARGET_EFFECTIVE_TOKENS,
     'corpus_raw_tokens':CORPUS_RAW_TOKENS,'corpus_documents':CORPUS_DOCUMENTS},
 'architectures':{k:asdict(v) for k,v in CONFIGS.items()},
 'results':RESULTS,'budget_projection':summary,
 'limitations':[
     'Synthetic random tokens; no real corpus I/O or data-loader throughput measured.',
     'A few optimizer updates are not a steady-state production benchmark.',
     'Chunked vocabulary cross entropy changes memory/performance relative to other implementations.',
     'Reference implementation may differ from prior smoke-test model kernels and optimizer.',
     'A100 multi-GPU scaling cannot be inferred from single-GPU throughput.',
     'EOS planning counts exclude train/validation/test holdouts and packing policy.',
     'Repeated passes over identical data are not equivalent to new unique training tokens.',
     'Chinchilla 20 tokens per parameter is a rough heuristic, not an exact optimum.',
     'Historical tokenizer identity is not proven.'
 ]}
path=OUTPUT/'a100_benchmark_budget_report_v2.json'
with path.open('x',encoding='utf-8') as f:json.dump(report,f,indent=2,ensure_ascii=False,default=str)
print('Saved:',path)

## Optional: production follow-up

For the next benchmark, replace synthetic `ids` with a **prefetched local NVMe or RAM cache** of actual packed corpus sequences, keeping all other parameters identical. Benchmark separately (A) compute-only with prefetched batches and (B) end-to-end loader + compute + checkpoint I/O. Never compare end-to-end and compute-only throughput as if they were identical. Save a new report with both measurements, and compare loss curves across PT/EN/ES before deciding on 1 vs 2 corpus passes.